# EcoSort AI Classifier

## Data Collection and Dataset preparation
Firstly, we have to curate a dataset from different datasets and tailor it to our needs.
We are going to look at and download these datasets:
1. **Recyclable**: TrashNet (Gary Thung & Ming-Sheng Yang) and the TACO Dataset (`Trash Annotations in Context`). Target items: plastic bottles, aluminium beverage cans, cardboard boxes, clean glass jars.
2. **Wet (Organic)**: Source from the Waste Classification Dataset (Kaggle - `Techsash`) under category `O` (Organic) or the Food Waste Image Dataset. Target items: vegetable peels, fruit rinds, leftover food, tea bags, fallen leaves.
3. **E-waste**: Source from the Electronic Waste Classification Dataset (Kaggle) or E-Waste Image Dataset. Target items: printed circuit boards (PCBs), discarded smartphones, damaged cables/chargers, computer mice, keyboards, batteries.
4. **Dry (Non-recyclable / Residual)**: Source from the trash category of TrashNet and general residual waste subsets. Target items: soiled tissues, plastic cutlery, chip/snack multi-layer laminate wrappers, polystyrene/Styrofoam packaging.

**Note:** Our dataset is highly **_skewed_**.
1. `ewaste` has 3,002 images
2. `dry` has 702 images
3. wet` has 17,350 images
4. `recyclable` has 16,279 images
Upon considering these details, we must focus on precision/recall and F1 score, and we are not going to blindly follow the Accuracy metric.

### Import required libraries

In [5]:
# import torch and torchvision
import torch
import torchvision
print(torch.__version__, torchvision.__version__)

2.12.1+cu130 0.27.1+cu130


In [6]:
# continue with regular imports
from torch import nn
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torchinfo import summary
import data_setup, engine

In [7]:
# Set up device-agnostic code
device ='cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

### Get Data
Get the  dry, ewaste, recyclable, wet waste data to train our models.

In [8]:
from pathlib import Path
data_path = Path('data/')
data_path

WindowsPath('data')

In [9]:
# Set up train, cross val and test dir
train_dir = data_path / 'train'
val_dir = data_path / 'val'
test_dir = data_path / 'test'
train_dir, val_dir, test_dir

(WindowsPath('data/train'), WindowsPath('data/val'), WindowsPath('data/test'))

### Transform the data
Using `torchvision.transforms.v2` we shall transform the dataset and prepare dataloaders for our first model: the baseline model.

In [20]:
import torchvision.transforms.v2 as transforms
from torchvision import datasets
# Create a simple transform for our baseline model
data_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToImage(),
    transforms.ToDtype(torch.float32, scale=True),
])
# Use the `create_dataloaders()` function from data_setup.py file
train_dataloader, test_dataloader, class_names = data_setup.create_dataloaders(train_dir,
                                                                               test_dir,
                                                                               data_transform,
                                                                               batch_size=64,
                                                                               num_workers=4)

In [21]:
train_dataloader, test_dataloader, class_names

(<torch.utils.data.dataloader.DataLoader at 0x2332c495ba0>,
 ['dry', 'ewaste', 'recyclable', 'wet'])

In [22]:
# Test the dataloaders
next(iter(train_dataloader))

[tensor([[[[1.0000, 1.0000, 1.0000,  ..., 1.0000, 1.0000, 1.0000],
           [1.0000, 1.0000, 1.0000,  ..., 1.0000, 1.0000, 1.0000],
           [1.0000, 1.0000, 1.0000,  ..., 1.0000, 1.0000, 1.0000],
           ...,
           [1.0000, 1.0000, 1.0000,  ..., 1.0000, 1.0000, 1.0000],
           [1.0000, 1.0000, 1.0000,  ..., 1.0000, 1.0000, 1.0000],
           [1.0000, 1.0000, 1.0000,  ..., 1.0000, 1.0000, 1.0000]],
 
          [[1.0000, 1.0000, 1.0000,  ..., 1.0000, 1.0000, 1.0000],
           [1.0000, 1.0000, 1.0000,  ..., 1.0000, 1.0000, 1.0000],
           [1.0000, 1.0000, 1.0000,  ..., 1.0000, 1.0000, 1.0000],
           ...,
           [1.0000, 1.0000, 1.0000,  ..., 1.0000, 1.0000, 1.0000],
           [1.0000, 1.0000, 1.0000,  ..., 1.0000, 1.0000, 1.0000],
           [1.0000, 1.0000, 1.0000,  ..., 1.0000, 1.0000, 1.0000]],
 
          [[1.0000, 1.0000, 1.0000,  ..., 1.0000, 1.0000, 1.0000],
           [1.0000, 1.0000, 1.0000,  ..., 1.0000, 1.0000, 1.0000],
           [1.0000, 1.00

## Making Classifiers to classify the data
We are going to make three models: a baseline, feature-extracted pretrained CNN model, and a pretrained transformer model.
### 1. Baseline Model
The baseline model is a simple CNN model that we shall train to classify the data.
For the baseline model we are going to use the ResNet-18 architecture. Source:  [ResNet-18 Architecture](https://www.geeksforgeeks.org/deep-learning/resnet18-from-scratch-using-pytorch/)

In [23]:
class BasicBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        out += self.shortcut(x)
        out = self.relu(out)
        return out

In [27]:
class ResNet18(nn.Module):
    def __init__(self, num_classes=len(class_names)):
        super().__init__()
        self.in_channels = 64
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)

        self.layer1 = self._make_layer(BasicBlock, 64, 2, stride=1)
        self.layer2 = self._make_layer(BasicBlock, 128, 2, stride=2)
        self.layer3 = self._make_layer(BasicBlock, 256, 2, stride=2)
        self.layer4 = self._make_layer(BasicBlock, 512, 2, stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, num_classes)

    def _make_layer(self, block, out_channels, num_blocks, stride):
        strides = [stride] + [1]*(num_blocks-1)
        layers = []
        for stride in strides:
            layers.append(block(self.in_channels, out_channels, stride))
            self.in_channels = out_channels
        return nn.Sequential(*layers)

    def forward(self, x):
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)

        out = self.avgpool(out)
        out = out.view(out.size(0), -1)
        out = self.fc(out)
        return out

model_0 = ResNet18().to(device)
print(model)

ResNet18(
  (conv1): Conv2d(1, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (shortcut): Sequential()
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2)

In [28]:
# 1. Get a batch of images and labels from the DataLoader
img_batch, label_batch = next(iter(train_dataloader))

# 2. Get a single image from the batch and unsqueeze the image so its shape fits the model
img_single, label_single = img_batch[0].unsqueeze(dim=0), label_batch[0]
print(f"Single image shape: {img_single.shape}\n")

# 3. Perform a forward pass on a single image
model_0.eval()
with torch.inference_mode():
    pred = model_0(img_single.to(device))

# 4. Print out what's happening and convert model logits -> pred probs -> pred label
print(f"Output logits:\n{pred}\n")
print(f"Output prediction probabilities:\n{torch.softmax(pred, dim=1)}\n")
print(f"Output prediction label:\n{torch.argmax(torch.softmax(pred, dim=1), dim=1)}\n")
print(f"Actual label:\n{label_single}")

Single image shape: torch.Size([1, 3, 64, 64])

Output logits:
tensor([[0.0258, 0.0354, 0.0260, 0.0250]], device='cuda:0')

Output prediction probabilities:
tensor([[0.2494, 0.2518, 0.2495, 0.2492]], device='cuda:0')

Output prediction label:
tensor([1], device='cuda:0')

Actual label:
2


### Training the BaseLine
Let us train the model above using training scripts.

In [29]:
from engine import train
# Set random seeds
torch.manual_seed(42)
torch.cuda.manual_seed(42)

# Set the number of epochs
NUM_EPOCHS = 5

# Setup loss function and optimizer
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=model_0.parameters(), lr=0.001)

# Start the timer
from timeit import default_timer as timer
start_time = timer()

# Train model_0
model_0_results = train(model=model_0,
                        train_dataloader=train_dataloader,
                        test_dataloader=test_dataloader,
                        optimizer=optimizer,
                        loss_fn=loss_fn,
                        epochs=NUM_EPOCHS,
                        device=device)

# End the timer and print out how long it took
end_time = timer()
print(f"[INFO] Total training time: {end_time-start_time:.3f} seconds")

  0%|          | 0/5 [00:00<?, ?it/s]

Epoch: 1 | train_loss: 0.6688 | train_acc: 0.7409 | test_loss: 0.6277 | test_acc: 0.7623
Epoch: 2 | train_loss: 0.5622 | train_acc: 0.7847 | test_loss: 0.5141 | test_acc: 0.7936
Epoch: 3 | train_loss: 0.5231 | train_acc: 0.8016 | test_loss: 0.7748 | test_acc: 0.7197
Epoch: 4 | train_loss: 0.4914 | train_acc: 0.8110 | test_loss: 0.4896 | test_acc: 0.8089
Epoch: 5 | train_loss: 0.4733 | train_acc: 0.8210 | test_loss: 0.4549 | test_acc: 0.8311
[INFO] Total training time: 367.111 seconds
